# 0. Setup

This notebook prepares the workspace for local, Colab, and Kaggle runs. It keeps data outside Git, checks GPU availability when requested, and writes a small status file used by the next notebooks.

In [1]:
from pathlib import Path
import os
import sys
import shutil
import subprocess
import zipfile
import json

REPO_NAME = 'ML-Tubes-2_RecursiveLearnaholic'
REPO_URL = 'https://github.com/Rusmn/ML-Tubes-2_RecursiveLearnaholic.git'
BRANCH = None
GPU_REQUIRED = None
USE_SYMLINK = True
DOWNLOAD_DATA = True

KAGGLE_WORK = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')
COLAB_WORK = Path('/content')


def detect_platform():
    if KAGGLE_WORK.exists():
        return 'kaggle', KAGGLE_WORK, KAGGLE_INPUT
    if COLAB_WORK.exists():
        return 'colab', COLAB_WORK, COLAB_WORK
    return 'local', Path.cwd().resolve(), Path.cwd().resolve()


def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    return None

PLATFORM, WORK_DIR, INPUT_DIR = detect_platform()
if GPU_REQUIRED is None:
    GPU_REQUIRED = PLATFORM in {'kaggle', 'colab'}
ROOT = find_root()
if ROOT is None and PLATFORM in {'kaggle', 'colab'}:
    ROOT = WORK_DIR / REPO_NAME
    if not (ROOT / 'src').exists():
        cmd = ['git', 'clone', '--depth', '1']
        if BRANCH:
            cmd += ['--branch', BRANCH]
        cmd += [REPO_URL, str(ROOT)]
        subprocess.run(cmd, check=True)
elif ROOT is None:
    raise RuntimeError('Repo root tidak ditemukan. Jalankan notebook dari dalam repo.')

ROOT = ROOT.resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('platform:', PLATFORM)
print('root:', ROOT)
print('input:', INPUT_DIR)

platform: local
root: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic
input: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/notebooks


In [2]:
import importlib

required = ['numpy', 'pandas', 'matplotlib', 'tensorflow', 'sklearn', 'PIL']
missing = []
for package in required:
    try:
        importlib.import_module('PIL' if package == 'PIL' else package)
        print(package, 'ok')
    except Exception as exc:
        print(package, 'missing:', exc)
        missing.append(package)

if missing:
    print('Install dependency yang hilang sebelum lanjut:', missing)

numpy ok
pandas ok
matplotlib ok
tensorflow ok
sklearn ok
PIL ok


In [3]:
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print('GPU:', gpus)
    if GPU_REQUIRED and not gpus:
        raise RuntimeError('GPU tidak terdeteksi.')
except ImportError:
    if GPU_REQUIRED:
        raise
    print('TensorFlow belum tersedia.')

GPU: []


## Data Setup

The cell below searches attached inputs first. If the data is not found and download is enabled, it uses Kaggle public datasets. On local runs it usually just detects the existing `data/raw` folders.

In [4]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}
INTEL_CLASSES = {'buildings', 'forest', 'glacier', 'mountain', 'sea', 'street'}


def count_images(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for item in path.rglob('*') if item.suffix.lower() in IMAGE_EXTS)


def class_dirs(path):
    path = Path(path)
    if not path.exists():
        return set()
    return {item.name for item in path.iterdir() if item.is_dir()}


def find_intel(search_root):
    search_root = Path(search_root)
    if not search_root.exists():
        return None, None
    candidates = []
    for path in search_root.rglob('*'):
        if not path.is_dir() or path.name.lower() not in {'seg_train', 'seg_test'}:
            continue
        nested = path / path.name
        if INTEL_CLASSES.issubset(class_dirs(path)):
            candidates.append((path.name.lower(), path, count_images(path)))
        elif nested.exists() and INTEL_CLASSES.issubset(class_dirs(nested)):
            candidates.append((path.name.lower(), nested, count_images(nested)))
    train = sorted([item for item in candidates if item[0] == 'seg_train'], key=lambda item: -item[2])
    test = sorted([item for item in candidates if item[0] == 'seg_test'], key=lambda item: -item[2])
    return (train[0][1] if train else None), (test[0][1] if test else None)


def find_flickr(search_root):
    search_root = Path(search_root)
    if not search_root.exists():
        return None, None
    image_dirs = []
    captions = []
    for path in search_root.rglob('*'):
        if path.is_file() and path.name.lower() in {'captions.txt', 'flickr8k.token.txt'}:
            captions.append(path)
        elif path.is_dir() and path.name.lower() in {'images', 'flicker8k_dataset', 'flickr8k_dataset'}:
            total = count_images(path)
            if total > 1000:
                image_dirs.append((path, total))
    image_dirs = sorted(image_dirs, key=lambda item: -item[1])
    captions = sorted(captions, key=lambda path: (path.name.lower() != 'captions.txt', len(str(path))))
    return (image_dirs[0][0] if image_dirs else None), (captions[0] if captions else None)


def extract_zips(search_root, extract_root):
    search_root = Path(search_root)
    extract_root = Path(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    if not search_root.exists():
        return []
    outputs = []
    for zip_path in search_root.rglob('*.zip'):
        name = zip_path.name.lower()
        if not any(key in name for key in ['intel', 'seg', 'flickr', 'flicker', 'caption']):
            continue
        target = extract_root / zip_path.stem
        marker = extract_root / f'{zip_path.stem}.done'
        if not marker.exists():
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(target)
            marker.write_text('done', encoding='utf-8')
        outputs.append(target)
    return outputs


def download_dataset(slug, target_root):
    target_root = Path(target_root)
    target_root.mkdir(parents=True, exist_ok=True)
    target = target_root / slug.replace('/', '__')
    if target.exists():
        return target
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(slug))
        try:
            os.symlink(downloaded, target, target_is_directory=True)
        except OSError:
            shutil.copytree(downloaded, target)
        return target
    except Exception:
        target.mkdir(parents=True, exist_ok=True)
        subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(target), '--unzip'], check=True)
        return target


def place(src, dst):
    src = Path(src).resolve()
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return dst
    if USE_SYMLINK:
        try:
            os.symlink(src, dst, target_is_directory=src.is_dir())
            return dst
        except OSError:
            pass
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return dst

search_roots = [ROOT / 'data' / 'raw', INPUT_DIR]
extract_root = WORK_DIR / 'datasets_extracted'
for base in list(search_roots):
    search_roots.extend(extract_zips(base, extract_root))

intel_train = intel_test = flickr_images = flickr_captions = None
for base in search_roots:
    intel_train = intel_train or find_intel(base)[0]
    intel_test = intel_test or find_intel(base)[1]
    flickr_images = flickr_images or find_flickr(base)[0]
    flickr_captions = flickr_captions or find_flickr(base)[1]

if DOWNLOAD_DATA and (intel_train is None or intel_test is None):
    data_root = WORK_DIR / 'kaggle_datasets'
    download_dataset('puneet6060/intel-image-classification', data_root)
    intel_train = intel_train or find_intel(data_root)[0]
    intel_test = intel_test or find_intel(data_root)[1]

if DOWNLOAD_DATA and (flickr_images is None or flickr_captions is None):
    data_root = WORK_DIR / 'kaggle_datasets'
    download_dataset('adityajn105/flickr8k', data_root)
    flickr_images = flickr_images or find_flickr(data_root)[0]
    flickr_captions = flickr_captions or find_flickr(data_root)[1]

missing = []
if intel_train is None: missing.append('Intel train')
if intel_test is None: missing.append('Intel test')
if flickr_images is None: missing.append('Flickr images')
if flickr_captions is None: missing.append('Flickr captions')
if missing:
    raise FileNotFoundError(', '.join(missing))

paths = {
    'intel_train': str(place(intel_train, ROOT / 'data/raw/intel/seg_train')),
    'intel_test': str(place(intel_test, ROOT / 'data/raw/intel/seg_test')),
    'flickr_images': str(place(flickr_images, ROOT / 'data/raw/flickr8k/Images')),
    'flickr_captions': str(place(flickr_captions, ROOT / 'data/raw/flickr8k/captions.txt')),
}
status = {
    'platform': PLATFORM,
    'root': str(ROOT),
    'paths': paths,
    'counts': {
        'intel_train': count_images(paths['intel_train']),
        'intel_test': count_images(paths['intel_test']),
        'flickr_images': count_images(paths['flickr_images']),
    },
}
status_path = ROOT / 'reports/tables/setup_status.json'
status_path.parent.mkdir(parents=True, exist_ok=True)
status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
print(json.dumps(status, indent=2))

{
  "platform": "local",
  "root": "/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic",
  "paths": {
    "intel_train": "/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/data/raw/intel/seg_train",
    "intel_test": "/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/data/raw/intel/seg_test",
    "flickr_images": "/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/data/raw/flickr8k/Images",
    "flickr_captions": "/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/data/raw/flickr8k/captions.txt"
  },
  "counts": {
    "intel_train": 14034,
    "intel_test": 3000,
    "flickr_images": 8091
  }
}
